# SEN12MS-CR 학습 노트북

이 노트북은 `main.py`와 같은 학습 흐름을 그대로 쓰되, 배치 확인과 결과 시각화만 노트북에 맞게 덧붙인 버전이다.

Colab에서는 저장소가 없으면 레포를 먼저 clone하고, 의존성 설치는 `uv`로 맞춘 뒤 같은 `main.py` helper를 재사용한다.

- 실행 환경 준비
- Colab 저장소 준비 및 `uv` 설치
- 학습 설정값 지정
- `main.py` helper 재사용
- 배치 shape 확인
- 학습 로그 표/그래프 확인
- 복원 결과 예시 저장 및 시각화


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
HF_TOKEN_INPUT = ""
TORCH_EXTRA = "torch26"
TORCH_SPEC = "torch==2.6.*" if TORCH_EXTRA == "torch26" else "torch>=2.11"

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir('/content/drive/MyDrive/cr_project')

WORKDIR = Path.cwd()
if not (WORKDIR / "pyproject.toml").exists():
    raise FileNotFoundError("cr_project 루트에서 실행해야 합니다.")
if TORCH_EXTRA not in {"torch26", "latest"}:
    raise ValueError("TORCH_EXTRA는 'torch26' 또는 'latest'여야 합니다.")

# pip로 cr-train과 torch만 설치해서 notebook 실행 환경을 맞춘다.
# cr-train commit은 현재 pyproject.toml과 동일한 버전으로 고정한다.
CR_TRAIN_REV = "f64603a318397777ffb0d650cff99a9516fd745a"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
      f"git+https://github.com/smturtle2/cr-train.git@{CR_TRAIN_REV}",
      TORCH_SPEC,
      "matplotlib", "pandas", "huggingface_hub"],
    check=True,
)

OUTPUT_DIR = Path("/content/drive/MyDrive/cr_project/artifacts") if IN_COLAB else WORKDIR / "artifacts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

token = os.environ.get("HF_TOKEN", "").strip() or HF_TOKEN_INPUT.strip()
if token:
    os.environ["HF_TOKEN"] = token

print(f"IN_COLAB={IN_COLAB}")
print(f"WORKDIR={WORKDIR}")
print(f"TORCH_EXTRA={TORCH_EXTRA}")
print(f"OUTPUT_DIR={OUTPUT_DIR}")
print(f"HF_TOKEN={'configured' if token else 'not found'}")


In [ ]:
import importlib
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display
from torch import nn
from cr_train import Trainer

# 긴 보조 함수는 notebook에 다시 복붙하지 않고 main.py에서 그대로 가져온다.
# 이렇게 두면 스크립트와 노트북이 같은 로직을 공유하므로 동작 차이가 줄어든다.
import main as train_app
train_app = importlib.reload(train_app)

# 아래 값들만 바꾸면 같은 helper를 써서 다양한 실험을 반복할 수 있다.
batch_size = 4
seed = 42
max_epochs = 20
train_max_samples = 4096
val_max_samples = 512
test_max_samples = 512
resume = None
run_test = False
num_examples = 4
example_stage = "val"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}")
print(f"output_dir={OUTPUT_DIR}")


In [ ]:
# modules/ 하위의 custom 구현을 사용
from modules.model import build_model as _build_model
from modules.optimizer import build_optimizer as _build_optimizer
from modules.loss_fn import build_loss as _build_loss
from modules.metrics import build_metrics as _build_metrics


def build_model() -> nn.Module:
    # FreqACACRNet: ACA-CRNet baseline + AFR-CR Frequency Fusion
    return _build_model(sar_channels=2, opt_channels=13, base_channels=64)


def build_optimizer(model: nn.Module) -> torch.optim.Optimizer:
    # AdamW with weight decay (주파수 영역 학습 안정화)
    return _build_optimizer(model, lr=1e-4, weight_decay=1e-4)


def build_loss():
    # L1 (공간 도메인) + Frequency Loss (주파수 도메인) 하이브리드
    return _build_loss(freq_weight=0.1, phase_weight=0.1)


def build_metrics() -> dict[str, object]:
    # MAE + PSNR + SSIM + SAM (SEN12MS-CR 표준)
    return _build_metrics()


In [ ]:
# 모델 초기화 재현성을 위해 Trainer 생성 전에 시드를 먼저 고정한다.
train_app.seed_everything(seed)
train_app.print_hf_auth_status()

# 모델과 optimizer는 프로젝트 쪽 구현만 연결하면 된다.
model = build_model().to(device)
optimizer = build_optimizer(model)

# Trainer 생성은 직접 두되, loss와 metric은 build 함수로 분리해서
# 프로젝트별 교체 지점이 어디인지 노트북에서도 똑같이 보이게 한다.
# 그래서 thin wrapper는 없애고 build 쪽만 남긴다.
# Trainer가 어떤 설정으로 생성되는지 셀에서 바로 보이도록 여기서 직접 만든다.
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    loss=build_loss(),
    metrics=build_metrics(),
    max_train_samples=train_max_samples,
    max_val_samples=val_max_samples,
    max_test_samples=test_max_samples,
    batch_size=batch_size,
    epochs=max_epochs,
    seed=seed,
    output_dir=OUTPUT_DIR,
)

# checkpoint에서 이어서 시작할 때도 main.py helper를 그대로 쓴다.
if resume is not None:
    train_app.load_checkpoint(trainer, Path(resume), device=device)

# 첫 train batch를 한 번 확인해 두면 모델 입력 shape를 바로 검증할 수 있다.
train_loader = train_app.build_loader(
    trainer,
    split="train",
    max_samples=train_max_samples,
    training=True,
    epoch_index=trainer.current_epoch,
)
sample_batch = next(iter(train_loader))

sar = sample_batch["sar"]
cloudy = sample_batch["cloudy"]
target = sample_batch["target"]
metadata = sample_batch.get("meta", {})

print(f"sar     shape={tuple(sar.shape)}  dtype={sar.dtype}  range=[{sar.min():.3f}, {sar.max():.3f}]")
print(f"cloudy  shape={tuple(cloudy.shape)}  dtype={cloudy.dtype}  range=[{cloudy.min():.3f}, {cloudy.max():.3f}]")
print(f"target  shape={tuple(target.shape)}  dtype={target.dtype}  range=[{target.min():.3f}, {target.max():.3f}]")
print(f"metadata keys: {list(metadata.keys())}")


In [ ]:
# Trainer 객체 상태를 셀에서 바로 확인할 수 있게 따로 남겨 둔다.
trainer


In [ ]:
# Trainer.step()은 매 epoch마다 epoch-XXXX.pt를 자동 저장한다.
# 여기서는 val_loss가 "역대 최저"를 갱신한 epoch만 남기고(이미지+가중치 누적 보존),
# 개선되지 않은 epoch의 체크포인트는 바로 삭제한다.
# 예: epoch1=best → 유지, epoch2 악화 → 삭제, epoch3이 epoch1보다 좋음 → 유지
# 결과 폴더에는 epoch1, epoch3의 가중치와 이미지가 모두 남는다.
history = []
best_val_loss: float | None = None

while trainer.current_epoch < trainer.epochs:
    record = trainer.step()
    history.append(train_app.flatten_record(record, global_step=trainer.global_step))

    epoch_idx = trainer.current_epoch  # step() 이후라 이미 +1 된 상태
    current_val_loss = float(record["val"]["loss"])
    current_ckpt = Path(str(record["checkpoint_path"]))

    improved = (best_val_loss is None) or (current_val_loss < best_val_loss)
    if improved:
        best_val_loss = current_val_loss
        # 이번 epoch의 복원 예시 이미지 저장
        train_app.save_examples_for_epoch(
            trainer,
            device=device,
            split="validation" if example_stage == "val" else "test",
            max_samples=val_max_samples if example_stage == "val" else test_max_samples,
            stage=example_stage,
            epoch=epoch_idx,
            output_dir=OUTPUT_DIR / "examples" / f"epoch_{epoch_idx:03d}",
            num_examples=num_examples,
        )
        print(f"[epoch {epoch_idx:03d}] val_loss improved → {current_val_loss:.6f}  (kept {current_ckpt.name})")
    else:
        # 역대 최저를 못 넘겼으면 방금 저장된 체크포인트 삭제
        if current_ckpt.exists():
            current_ckpt.unlink()
        print(f"[epoch {epoch_idx:03d}] val_loss {current_val_loss:.6f} (best {best_val_loss:.6f}) — discarded {current_ckpt.name}")

# 학습 곡선과 표를 파일/노트북 양쪽에 남긴다.
train_app.save_history_plot(history, OUTPUT_DIR / "history.png")
history_df = pd.DataFrame(history)
if not history_df.empty:
    display(history_df)

# 필요할 때만 test split 평가를 추가로 실행한다.
if run_test:
    test_summary = trainer.test()
    test_row = {}
    if "loss" in test_summary:
        test_row["test_loss"] = float(test_summary["loss"])
    for name, value in test_summary.get("metrics", {}).items():
        test_row[f"test_{name}"] = float(value)
    test_df = pd.DataFrame([test_row])
    display(test_df)
else:
    test_df = None

print(f"final best val_loss: {best_val_loss:.6f}")
